# Faz 9 — Word Dosyası Oluşturma

Bu notebook `python-docx` kullanarak iki makale taslağını Word formatında üretir.

**Önkoşul:** Faz8_Makale_Analiz.ipynb çalıştırılmış olmalı (figürler `paper_output/figures/` altında bulunmalı).

**Çıktılar:**
- `paper_output/Makale1_Benchmark.docx`
- `paper_output/Makale2_Method.docx`

In [1]:
import os, sys
from pathlib import Path

IS_COLAB  = 'google.colab' in sys.modules
IS_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None
IS_LOCAL  = not IS_COLAB and not IS_KAGGLE

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/abdomen')
elif IS_KAGGLE:
    PROJECT_ROOT = Path('/kaggle/working/abdomen')
else:  # LOCAL
    import platform
    if platform.system() == 'Windows':
        PROJECT_ROOT = Path(r'D:/makale-pdf/Proje/abdomen')
    else:  # macOS / Linux
        PROJECT_ROOT = Path('/Users/ramazanpolat/Desktop/datasets/abdomen')

WORK_DIR  = PROJECT_ROOT if IS_LOCAL else Path('/content' if IS_COLAB else '/kaggle/working')
PAPER_DIR = WORK_DIR / 'paper_output'
FIG_DIR   = PAPER_DIR / 'figures'
TABLE_DIR = PAPER_DIR / 'tables'
PAPER_DIR.mkdir(parents=True, exist_ok=True)

print(f'PAPER_DIR: {PAPER_DIR}')

PAPER_DIR: D:\makale-pdf\Proje\abdomen\paper_output


In [2]:
try:
    import docx
    print('python-docx mevcut:', docx.__version__)
except ImportError:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'python-docx'], check=True)
    import docx
    print('python-docx yüklendi.')

python-docx yüklendi.


In [3]:
import pandas as pd
from docx import Document
from docx.shared import Inches, Pt, RGBColor, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

def setup_doc(title):
    doc = Document()
    section = doc.sections[0]
    section.page_width  = Cm(21.0)
    section.page_height = Cm(29.7)
    section.left_margin   = Cm(2.5)
    section.right_margin  = Cm(2.5)
    section.top_margin    = Cm(2.5)
    section.bottom_margin = Cm(2.5)
    styles = doc.styles
    normal = styles['Normal']
    normal.font.name = 'Times New Roman'
    normal.font.size = Pt(12)
    return doc

def add_title(doc, text):
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = p.add_run(text)
    run.bold = True
    run.font.size = Pt(16)
    run.font.name = 'Times New Roman'

def add_authors(doc, authors, affiliations):
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = p.add_run(authors)
    run.font.size = Pt(11)
    run.font.name = 'Times New Roman'
    p2 = doc.add_paragraph()
    p2.alignment = WD_ALIGN_PARAGRAPH.CENTER
    r2 = p2.add_run(affiliations)
    r2.font.size = Pt(10)
    r2.font.name = 'Times New Roman'
    r2.italic = True

def add_heading(doc, text, level=1):
    p = doc.add_heading(text, level=level)
    for run in p.runs:
        run.font.name = 'Times New Roman'
    return p

def add_para(doc, text, bold=False, italic=False, font_size=12):
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
    run = p.add_run(text)
    run.bold = bold
    run.italic = italic
    run.font.size = Pt(font_size)
    run.font.name = 'Times New Roman'

def add_figure(doc, fig_path, caption, width_inches=5.5):
    p_img = doc.add_paragraph()
    p_img.alignment = WD_ALIGN_PARAGRAPH.CENTER
    if Path(fig_path).exists():
        run = p_img.add_run()
        run.add_picture(str(fig_path), width=Inches(width_inches))
    else:
        run = p_img.add_run(f'[FIGURE NOT FOUND: {Path(fig_path).name}]')
        run.font.color.rgb = RGBColor(0xFF, 0x00, 0x00)
    p_cap = doc.add_paragraph()
    p_cap.alignment = WD_ALIGN_PARAGRAPH.CENTER
    r = p_cap.add_run(caption)
    r.bold = True
    r.font.size = Pt(10)
    r.font.name = 'Times New Roman'

def add_table_from_csv(doc, csv_path, caption):
    if not Path(csv_path).exists():
        add_para(doc, f'[TABLE NOT FOUND: {Path(csv_path).name}]', bold=True)
        return
    df = pd.read_csv(csv_path, index_col=0).round(4)
    n_rows, n_cols = df.shape
    table = doc.add_table(rows=n_rows+1, cols=n_cols+1)
    table.style = 'Table Grid'
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    # Header
    hdr = table.rows[0].cells
    hdr[0].text = df.index.name or ''
    for j, col in enumerate(df.columns):
        hdr[j+1].text = str(col)
    for cell in hdr:
        for par in cell.paragraphs:
            for run in par.runs:
                run.bold = True
                run.font.size = Pt(9)
    # Data
    for i, (idx, row) in enumerate(df.iterrows()):
        cells = table.rows[i+1].cells
        cells[0].text = str(idx)
        for j, val in enumerate(row):
            cells[j+1].text = str(val)
        for cell in cells:
            for par in cell.paragraphs:
                for run in par.runs:
                    run.font.size = Pt(9)
    p_cap = doc.add_paragraph()
    p_cap.alignment = WD_ALIGN_PARAGRAPH.CENTER
    r = p_cap.add_run(caption)
    r.bold = True
    r.font.size = Pt(10)
    r.font.name = 'Times New Roman'

print('Yardımcı fonksiyonlar hazır.')

Yardımcı fonksiyonlar hazır.


## Makale 1: Benchmark Paper

In [4]:
doc1 = setup_doc('Makale1_Benchmark')

# ── Başlık & Yazarlar ─────────────────────────────────────────────────────
add_title(doc1,
    'A Multi-Class CT Benchmark for Emergency Abdominal Pathology Detection:\n'
    'Dataset, Protocol, and Baseline Results'
)
doc1.add_paragraph()
add_authors(doc1, authors='[Author Names]',
            affiliations='[Institution Name], [City], [Country]')
doc1.add_paragraph()

# ── Abstract ──────────────────────────────────────────────────────────────
add_heading(doc1, 'Abstract', level=1)
add_para(doc1,
    'We introduce an emergency abdominal CT dataset covering six acute pathology '
    'super-classes: acute cholecystitis, kidney/ureter stone, acute pancreatitis, '
    'aortic aneurysm/dissection, acute appendicitis, and acute diverticulitis. '
    'The dataset consists of [N] patient cases with bounding-box and boundary-slice '
    'annotations, split into training, validation, and external holdout sets. '
    'We benchmark four representative detection/segmentation architectures — '
    'YOLOv12, SwinTransformer, nnUNet, and MedNeXt — using a unified evaluation '
    'protocol based on Top-5 Mean F1 across IoU thresholds 0.1–0.9. '
    'Our results show that [best model] achieves the highest overall performance '
    'while [weakest class] remains the most challenging class across all methods. '
    'Dataset, splits, and evaluation code are publicly available at [repository URL].')
add_para(doc1, 'Keywords: abdominal CT, emergency radiology, object detection, benchmark, deep learning',
         italic=True, font_size=11)
doc1.add_paragraph()

# ── 1. Introduction ───────────────────────────────────────────────────────
add_heading(doc1, '1. Introduction', level=1)
add_para(doc1,
    'Emergency abdominal pathologies — including acute cholecystitis, urolithiasis, '
    'pancreatitis, aortic dissection, appendicitis, and diverticulitis — are among '
    'the most time-critical conditions encountered in radiology. Rapid and accurate '
    'diagnosis from CT images is essential to guide triage and treatment decisions. '
    'Despite significant advances in medical image analysis, the lack of a '
    'comprehensive, multi-class benchmark specifically targeting emergency abdominal '
    'CT findings has hindered systematic comparison of detection methods.')
add_para(doc1,
    'Existing datasets focus on single-organ or single-pathology detection '
    '(e.g., RSNA Abdominal Trauma, LiTS, KiTS), limiting cross-pathology '
    'evaluation. In this work, we address this gap by: (1) introducing a new '
    'multi-class emergency abdominal CT dataset with expert bounding-box annotations '
    'for six pathology super-classes; (2) defining a standardized evaluation protocol; '
    'and (3) providing baseline results for four state-of-the-art models.')

# ── 2. Dataset ────────────────────────────────────────────────────────────
add_heading(doc1, '2. Dataset', level=1)
add_heading(doc1, '2.1 Data Collection and Patient Demographics', level=2)
add_para(doc1,
    '[Describe data collection procedure: institution, time period, inclusion/exclusion '
    'criteria, ethics approval, patient demographics (age, sex distribution), '
    'CT scanner models and acquisition protocols. Include IRB/Ethics board details.]')

add_heading(doc1, '2.2 Annotation Protocol', level=2)
add_para(doc1,
    'Two board-certified radiologists independently annotated each CT scan. '
    'Annotations consist of 2D axis-aligned bounding boxes on individual slices, '
    'with each box labeled as one of six pathology super-classes. '
    'Discrepancies were resolved via consensus review. '
    'Additionally, boundary slices (first/last appearance of each pathology) '
    'were marked to support segmentation-based methods. '
    'Inter-annotator agreement was measured using [Cohen\'s Kappa / IoU].')

add_heading(doc1, '2.3 Class Definitions and Distribution', level=2)
add_para(doc1,
    'The six super-classes are defined based on ICD-10 coding and radiological '
    'consensus. Multiple raw labels may map to the same super-class. '
    'Figure 1 shows the per-class case distribution across training and validation splits.')
doc1.add_paragraph()
add_figure(doc1, FIG_DIR / 'fig1_dataset_statistics.png',
    'Figure 1. Dataset statistics. (a) Per-class case counts for training and '
    'validation splits. (b) Overall train/validation split ratio.')
doc1.add_paragraph()

add_heading(doc1, '2.4 Dataset Splits', level=2)
add_para(doc1,
    'Cases are split into training (80%), validation (10%), and external holdout '
    'test (10%) sets using stratified sampling. The split is performed at the '
    'patient level to prevent data leakage. A fixed random seed ensures reproducibility.')

# ── 3. Evaluation Protocol ────────────────────────────────────────────────
add_heading(doc1, '3. Evaluation Protocol', level=1)
add_para(doc1,
    'All models are evaluated using the Top-5 Mean F1 metric, defined as the '
    'mean F1 score computed across five IoU thresholds: {0.1, 0.3, 0.5, 0.7, 0.9}. '
    'For each threshold, per-class precision, recall, and F1 are computed using '
    'greedy bounding-box matching (by descending confidence score). '
    'The macro-averaged F1 across the six super-classes is reported as the primary metric.')

# ── 4. Baseline Methods ───────────────────────────────────────────────────
add_heading(doc1, '4. Baseline Methods', level=1)
add_heading(doc1, '4.1 YOLOv12', level=2)
add_para(doc1,
    'YOLOv12 is applied to 2D axial slices converted to PNG format. '
    'Input images are resized to 640×640 and the model is trained for 100 epochs '
    'with AdamW optimizer, initial learning rate 1e-3, and cosine annealing.')

add_heading(doc1, '4.2 SwinTransformer', level=2)
add_para(doc1,
    'Swin Transformer is adapted as a classification backbone for slice-level '
    'multi-label pathology prediction, pretrained on ImageNet-22K and fine-tuned '
    'with WeightedRandomSampler and per-class threshold tuning.')

add_heading(doc1, '4.3 nnUNet', level=2)
add_para(doc1,
    'nnUNet is configured in 2D mode using boundary-slice annotations for '
    'segmentation-based detection. Final detections are extracted from predicted '
    'segmentation masks via connected-component analysis.')

add_heading(doc1, '4.4 MedNeXt', level=2)
add_para(doc1,
    'MedNeXt extends the ConvNeXt design to 3D medical image segmentation. '
    'We use the MedNeXt-B variant with kernel size 3×3×3 and deep supervision, '
    'with volumetric inputs resampled to 1mm isotropic resolution.')

# ── 5. Experiments and Results ────────────────────────────────────────────
add_heading(doc1, '5. Experiments and Results', level=1)

add_heading(doc1, '5.1 Training Details', level=2)
add_para(doc1,
    'All experiments are conducted on [GPU details]. '
    'Training and evaluation use fold 0 for initial benchmarking. '
    'Figure 2 shows the training curves for all models.')
doc1.add_paragraph()
add_figure(doc1, FIG_DIR / 'fig2_training_curves.png',
    'Figure 2. Training curves: (a) training loss, (b) validation loss, '
    '(c) validation Top-5 Mean F1.')
doc1.add_paragraph()

add_heading(doc1, '5.2 Benchmark Results', level=2)
add_para(doc1,
    'Table 1 reports per-class and overall Top-5 Mean F1 for all four baseline models. '
    'Figure 3 provides a visual comparison and Figure 4 shows the per-class F1 heatmap.')
doc1.add_paragraph()
add_table_from_csv(doc1, TABLE_DIR / 'table1_benchmark.csv',
    'Table 1. Benchmark results: Top-5 Mean F1 for each model across all pathology classes.')
doc1.add_paragraph()
add_figure(doc1, FIG_DIR / 'fig3_model_comparison.png',
    'Figure 3. Overall Top-5 Mean F1 comparison across the four baseline models.')
doc1.add_paragraph()
add_figure(doc1, FIG_DIR / 'fig4_per_class_f1_heatmap.png',
    'Figure 4. Per-class F1 heatmap. Rows: pathology classes; Columns: models.')
doc1.add_paragraph()

add_heading(doc1, '5.3 Error Analysis', level=2)
add_para(doc1,
    'Figure 5 shows the per-class F1 breakdown for each model. '
    'Across all methods, [class X] consistently shows the lowest F1, likely due to '
    'its subtle CT appearance and low case count. '
    'In contrast, [class Y] achieves the highest F1, benefiting from distinctive '
    'morphological features. '
    'Figure 6 presents the per-class Precision-Recall curves, confirming that '
    'class imbalance is a key driver of performance variation.')
doc1.add_paragraph()
add_figure(doc1, FIG_DIR / 'fig6_error_analysis.png',
    'Figure 5. Per-class F1 error analysis for each baseline model. '
    'Green: F1 ≥ 0.70; Orange: 0.50 ≤ F1 < 0.70; Red: F1 < 0.50.')
doc1.add_paragraph()
add_figure(doc1, FIG_DIR / 'fig9_pr_curves.png',
    'Figure 6. Per-class Precision-Recall curves for all four baseline models. '
    'AP = Average Precision.')
doc1.add_paragraph()

add_heading(doc1, '5.4 Model Calibration', level=2)
add_para(doc1,
    'Figure 7 presents the reliability diagrams for each model. '
    'A well-calibrated model should have its confidence-accuracy curve close to '
    'the diagonal. Expected Calibration Error (ECE) is reported in each panel. '
    '[Model X] shows the best calibration with ECE=[value], while [Model Y] '
    'tends to be overconfident, motivating per-class threshold tuning.')
doc1.add_paragraph()
add_figure(doc1, FIG_DIR / 'fig10_calibration.png',
    'Figure 7. Reliability diagrams (calibration curves) for each model. '
    'ECE = Expected Calibration Error. Closer to diagonal = better calibrated.')
doc1.add_paragraph()

add_heading(doc1, '5.5 Model Efficiency', level=2)
add_para(doc1,
    'Table 2 and Figure 8 compare the four models in terms of computational cost. '
    'Parameter counts range from [min]M (YOLO) to [max]M (SwinTransformer). '
    'Inference time (ms per image on [GPU]) and peak GPU memory are also reported, '
    'providing practical guidance for deployment under real-time constraints.')
doc1.add_paragraph()
add_table_from_csv(doc1, TABLE_DIR / 'table_efficiency.csv',
    'Table 2. Model efficiency comparison: parameter count, inference time per image, '
    'and peak GPU memory.')
doc1.add_paragraph()
add_figure(doc1, FIG_DIR / 'fig8_efficiency.png',
    'Figure 8. Model efficiency comparison: (a) parameter count, '
    '(b) inference time (ms/img), (c) peak GPU memory (GB).')
doc1.add_paragraph()

# ── 6. Discussion ─────────────────────────────────────────────────────────
add_heading(doc1, '6. Discussion', level=1)
add_para(doc1,
    '[Discuss key findings: which model performed best and why, '
    'which pathology classes are harder and clinical reasons for this, '
    'calibration implications for clinical deployment, '
    'efficiency-accuracy trade-offs for each model, '
    'limitations of the current benchmark (single institution, limited external '
    'test set), and future directions (multi-center, ensemble methods, 3D integration).]')

# ── 7. Conclusion ─────────────────────────────────────────────────────────
add_heading(doc1, '7. Conclusion', level=1)
add_para(doc1,
    'We presented the first multi-class emergency abdominal CT benchmark '
    'covering six acute pathology super-classes. Our standardized evaluation '
    'protocol, efficiency analysis, and calibration assessment serve as a '
    'comprehensive reference for future method development. '
    'The dataset and code are available at [URL].')

# ── References ────────────────────────────────────────────────────────────
add_heading(doc1, 'References', level=1)
add_para(doc1, '[1] Wang et al. YOLOv12: ... (2024)', font_size=11)
add_para(doc1, '[2] Liu et al. Swin Transformer: Hierarchical Vision Transformer using Shifted Windows. ICCV 2021.', font_size=11)
add_para(doc1, '[3] Isensee et al. nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature Methods 2021.', font_size=11)
add_para(doc1, '[4] Roy et al. MedNeXt: Transformer-Driven Scaling of ConvNets for Medical Image Segmentation. MICCAI 2023.', font_size=11)

out_path1 = PAPER_DIR / 'Makale1_Benchmark.docx'
doc1.save(out_path1)
print(f'Makale 1 kaydedildi: {out_path1}')

Makale 1 kaydedildi: D:\makale-pdf\Proje\abdomen\paper_output\Makale1_Benchmark.docx


## Makale 2: Method Paper

In [5]:
doc2 = setup_doc('Makale2_Method')

# ── Başlık & Yazarlar ─────────────────────────────────────────────────────
add_title(doc2,
    'Addressing Class Imbalance in Emergency Abdominal CT Classification:\n'
    'WeightedRandomSampler, Per-Class Threshold Optimization, and 5-Fold Validation'
)
doc2.add_paragraph()
add_authors(doc2, authors='[Author Names]',
            affiliations='[Institution Name], [City], [Country]')
doc2.add_paragraph()

# ── Abstract ──────────────────────────────────────────────────────────────
add_heading(doc2, 'Abstract', level=1)
add_para(doc2,
    'Multi-label classification of emergency abdominal CT findings is challenged '
    'by significant class imbalance: rare pathologies such as aortic dissection '
    'and acute pancreatitis are far less represented than common findings. '
    'We propose a two-stage approach: (1) inverse-frequency WeightedRandomSampler '
    'at the slice level combined with Focal-BCE loss; and (2) per-class decision '
    'threshold optimization using F1-score maximization across 51 candidates. '
    'We validate the method via 5-fold cross-validation, conduct a backbone ablation '
    'across six architectures, and analyze model calibration via reliability diagrams. '
    'Our method improves Top-5 Mean F1 from [baseline] to [ours] (mean±std over 5 folds), '
    'with the largest gains on minority classes.')
add_para(doc2,
    'Keywords: class imbalance, multi-label classification, abdominal CT, '
    'weighted sampling, threshold tuning, cross-validation, calibration',
    italic=True, font_size=11)
doc2.add_paragraph()

# ── 1. Introduction ───────────────────────────────────────────────────────
add_heading(doc2, '1. Introduction', level=1)
add_para(doc2,
    'Class imbalance is a pervasive challenge in medical image classification. '
    'In emergency abdominal radiology, the frequency of pathologies varies widely: '
    'urolithiasis and cholecystitis are relatively common, whereas aortic dissection '
    'and acute pancreatitis are rare. Standard cross-entropy training causes models '
    'to optimize predominantly for majority classes while under-detecting minority ones.')
add_para(doc2,
    'In this paper, we address imbalance at two complementary levels: '
    '(1) the sampling level, via slice-level WeightedRandomSampler; '
    'and (2) the decision level, via per-class threshold optimization. '
    'We further analyze model calibration and validate with 5-fold cross-validation '
    'to ensure statistical reliability of the reported gains.')

# ── 2. Related Work ───────────────────────────────────────────────────────
add_heading(doc2, '2. Related Work', level=1)
add_heading(doc2, '2.1 Class Imbalance in Medical Imaging', level=2)
add_para(doc2,
    '[Review: oversampling (SMOTE), undersampling, class weights, '
    'focal loss (Lin et al. 2017), curriculum learning, ensemble methods.]')
add_heading(doc2, '2.2 Multi-Label Classification of Medical Images', level=2)
add_para(doc2,
    '[Review: CheXNet, CheXpert, multi-label chest X-ray work. '
    'Discuss co-occurring pathologies and label correlation.]')
add_heading(doc2, '2.3 Threshold Optimization and Calibration', level=2)
add_para(doc2,
    '[Review: Platt scaling, temperature scaling, per-class threshold search. '
    'Discuss why fixed 0.5 threshold is suboptimal for imbalanced multi-label '
    'problems, and the role of calibration in clinical deployment.]')

# ── 3. Method ─────────────────────────────────────────────────────────────
add_heading(doc2, '3. Method', level=1)
add_heading(doc2, '3.1 Problem Formulation', level=2)
add_para(doc2,
    'Given a CT slice image x ∈ R^(H×W×3), we predict a binary vector '
    'y ∈ {0,1}^C where C=6. The model f_θ → [0,1]^C produces sigmoid-activated '
    'logits thresholded per class: ŷ_c = 1 if f_θ(x)_c ≥ τ_c else 0.')
add_heading(doc2, '3.2 Inverse-Frequency WeightedRandomSampler', level=2)
add_para(doc2,
    'Let n_c denote the number of slices containing class c. '
    'For slice i: w_i = max_c (y_{i,c} / n_c). '
    'PyTorch WeightedRandomSampler draws batches proportional to w_i, '
    'producing approximately balanced class representation per batch.')
add_heading(doc2, '3.3 Focal-BCE Loss', level=2)
add_para(doc2,
    'Focal Binary Cross-Entropy with γ=2 and α=0.25 down-weights easy examples '
    'and focuses training on hard, misclassified samples, complementing the sampler.')
add_heading(doc2, '3.4 Per-Class Threshold Optimization', level=2)
add_para(doc2,
    'After training, per-class thresholds τ_c are optimized on the validation set: '
    'τ*_c = argmax_{τ∈[0.05,0.95]} F1_c(τ), evaluated at 51 candidate values. '
    'This post-hoc step requires no retraining.')
add_heading(doc2, '3.5 Backbone Architectures', level=2)
add_para(doc2,
    'We evaluate six backbone architectures: Swin-Base, ConvNeXt-Base, '
    'ConvNeXtV2-Base, ViT-Base/16, DINOv2-Base, and EVA02-Base, '
    'all pretrained on ImageNet-22K and fine-tuned end-to-end.')

# ── 4. Experiments ────────────────────────────────────────────────────────
add_heading(doc2, '4. Experiments', level=1)
add_heading(doc2, '4.1 Dataset and Evaluation', level=2)
add_para(doc2,
    'We use the emergency abdominal CT dataset from [Makale 1 citation]. '
    'Primary metric: Top-5 Mean F1 (mean F1 across IoU thresholds 0.1–0.9). '
    'For cross-validation, results are reported as mean ± std over 5 folds.')
add_heading(doc2, '4.2 Training Configuration', level=2)
add_para(doc2,
    'AdamW optimizer (lr=2e-4, weight_decay=1e-4), cosine LR schedule, '
    'batch size 16, 50 epochs, input size 384×384, standard ImageNet normalization. '
    'Augmentation: random horizontal flip, color jitter, rotation ±15°.')

add_heading(doc2, '4.3 Ablation: Loss and Sampler (Fold 0)', level=2)
add_para(doc2,
    'Table 1 and Figure 1 show incremental ablation across four configurations '
    'on fold 0. Each component is isolated to quantify its individual contribution.')
doc2.add_paragraph()
add_table_from_csv(doc2, TABLE_DIR / 'table2_ablation.csv',
    'Table 1. Ablation study: effect of loss function, WeightedRandomSampler, '
    'and per-class threshold tuning on Top-5 Mean F1 (fold 0).')
doc2.add_paragraph()
add_figure(doc2, FIG_DIR / 'fig7_ablation_sampler_threshold.png',
    'Figure 1. Ablation results. (a) Overall Top-5 Mean F1 per configuration. '
    '(b) Per-class F1 comparison across configurations.')
doc2.add_paragraph()

add_heading(doc2, '4.4 5-Fold Cross-Validation', level=2)
add_para(doc2,
    'To confirm the generalizability of our findings, all four configurations are '
    'evaluated across 5 folds. Table 2 reports mean ± std and per-fold F1. '
    'Figure 2 visualizes the per-fold performance and fold-to-fold variability.')
doc2.add_paragraph()
add_table_from_csv(doc2, TABLE_DIR / 'table_crossval.csv',
    'Table 2. 5-fold cross-validation results: Top-5 Mean F1 per fold and '
    'mean ± std for each configuration.')
doc2.add_paragraph()
add_figure(doc2, FIG_DIR / 'fig11_crossval.png',
    'Figure 2. 5-fold cross-validation. (a) Mean ± std across folds. '
    '(b) Per-fold F1 trajectory for each configuration.')
doc2.add_paragraph()

add_heading(doc2, '4.5 Backbone Ablation', level=2)
add_para(doc2,
    'Table 3 and Figure 3 compare six backbone architectures using the best '
    'configuration (FocalBCE + WeightedSampler + ThreshTuning).')
doc2.add_paragraph()
add_table_from_csv(doc2, TABLE_DIR / 'table3_backbone.csv',
    'Table 3. Backbone ablation: Top-5 Mean F1 across six pretrained backbone architectures.')
doc2.add_paragraph()
add_figure(doc2, FIG_DIR / 'fig5_backbone_ablation.png',
    'Figure 3. Backbone ablation study. (a) Overall Top-5 Mean F1. '
    '(b) Per-class F1 heatmap across backbone architectures.')
doc2.add_paragraph()

# ── 5. Results and Discussion ─────────────────────────────────────────────
add_heading(doc2, '5. Results and Discussion', level=1)

add_heading(doc2, '5.1 Effect of Sampler and Threshold Tuning', level=2)
add_para(doc2,
    'Our full method achieves Top-5 Mean F1 = [X.XXX ± Y.YYY] over 5 folds, '
    'outperforming the baseline ([A.AAA ± B.BBB]) by [delta]. '
    'The WeightedSampler contributes the largest single gain ([+delta_sampler]), '
    'particularly for minority classes (aortic_aneurysm_dissection: +[delta] F1). '
    'Threshold tuning provides an additional +[delta_thresh] without any retraining.')

add_heading(doc2, '5.2 Precision-Recall Analysis', level=2)
add_para(doc2,
    'Figure 4 shows per-class PR curves for all models, with the operating point '
    'marked before and after threshold tuning. The curves confirm that the default '
    '0.5 threshold is suboptimal for minority classes, and that per-class tuning '
    'moves the operating point closer to the Pareto frontier of precision-recall trade-off.')
doc2.add_paragraph()
add_figure(doc2, FIG_DIR / 'fig9_pr_curves.png',
    'Figure 4. Per-class Precision-Recall curves. AP = Average Precision per class.')
doc2.add_paragraph()

add_heading(doc2, '5.3 Calibration Analysis', level=2)
add_para(doc2,
    'Figure 5 shows reliability diagrams for each model configuration. '
    'The baseline (BCE) model exhibits overconfidence (ECE=[value]). '
    'Focal-BCE partially corrects this, and per-class threshold tuning further '
    'improves calibration by shifting decision boundaries toward better-calibrated '
    'operating points. The final ECE=[value] represents a [X]% reduction vs. baseline.')
doc2.add_paragraph()
add_figure(doc2, FIG_DIR / 'fig10_calibration.png',
    'Figure 5. Reliability diagrams. ECE = Expected Calibration Error. '
    'Diagonal = perfect calibration.')
doc2.add_paragraph()

add_heading(doc2, '5.4 Backbone Comparison', level=2)
add_para(doc2,
    'Among backbones, [best_backbone] achieves the highest overall F1 of [X.XXX]. '
    'EVA02 and DINOv2 benefit from large-scale self-supervised pretraining, '
    'which appears particularly advantageous for rare-class detection. '
    'ViT-Base/16 underperforms, likely due to limited positional inductive bias. '
    'Table 4 reports the efficiency metrics for each backbone.')
doc2.add_paragraph()
add_table_from_csv(doc2, TABLE_DIR / 'table_efficiency.csv',
    'Table 4. Efficiency comparison: parameter count, inference time, and VRAM for each model.')
doc2.add_paragraph()
add_figure(doc2, FIG_DIR / 'fig8_efficiency.png',
    'Figure 6. Efficiency comparison: (a) parameters, (b) inference time, (c) peak VRAM.')
doc2.add_paragraph()

# ── 6. Conclusion ─────────────────────────────────────────────────────────
add_heading(doc2, '6. Conclusion', level=1)
add_para(doc2,
    'We presented a two-stage approach for addressing class imbalance in '
    'multi-label emergency abdominal CT classification. '
    'Validated across 5 folds, our method (FocalBCE + WeightedSampler + ThreshTuning) '
    'consistently outperforms the baseline, with the most significant improvements '
    'for rare pathologies. Calibration analysis confirms that threshold tuning '
    'also improves decision reliability, which is critical for clinical deployment. '
    'Our code and pretrained models are available at [URL].')

# ── References ────────────────────────────────────────────────────────────
add_heading(doc2, 'References', level=1)
add_para(doc2, '[1] Lin et al. Focal Loss for Dense Object Detection. ICCV 2017.', font_size=11)
add_para(doc2, '[2] Liu et al. Swin Transformer. ICCV 2021.', font_size=11)
add_para(doc2, '[3] Woo et al. ConvNeXt V2. CVPR 2023.', font_size=11)
add_para(doc2, '[4] Oquab et al. DINOv2. TMLR 2024.', font_size=11)
add_para(doc2, '[5] Sun et al. EVA-02. arXiv 2023.', font_size=11)
add_para(doc2, '[Makale 1]: [Authors]. A Multi-Class CT Benchmark for Emergency Abdominal Pathology Detection. [Journal] 2026.', font_size=11)

out_path2 = PAPER_DIR / 'Makale2_Method.docx'
doc2.save(out_path2)
print(f'Makale 2 kaydedildi: {out_path2}')

Makale 2 kaydedildi: D:\makale-pdf\Proje\abdomen\paper_output\Makale2_Method.docx


In [6]:
print('=== Faz 9 Tamamlandı ===')
print(f'\n  {out_path1}')
print(f'  {out_path2}')
print(f'\nFigürler: {FIG_DIR}')
print(f'Tablolar: {TABLE_DIR}')
print('\nSONRAKİ ADIMLAR:')
print('  1. Modelleri çalıştır ve gerçek JSON sonuçlarını outputs/results/ klasörüne kaydet')
print('  2. Faz8_Makale_Analiz.ipynb yeniden çalıştır (gerçek figürler üretilir)')
print('  3. Faz9_Word_Olustur.ipynb yeniden çalıştır (figürler Word dosyalarına eklenir)')
print('  4. [PLACEHOLDER] ve [dummy data] bölümlerini gerçek bulgularla doldur')

=== Faz 9 Tamamlandı ===

  D:\makale-pdf\Proje\abdomen\paper_output\Makale1_Benchmark.docx
  D:\makale-pdf\Proje\abdomen\paper_output\Makale2_Method.docx

Figürler: D:\makale-pdf\Proje\abdomen\paper_output\figures
Tablolar: D:\makale-pdf\Proje\abdomen\paper_output\tables

SONRAKİ ADIMLAR:
  1. Modelleri çalıştır ve gerçek JSON sonuçlarını outputs/results/ klasörüne kaydet
  2. Faz8_Makale_Analiz.ipynb yeniden çalıştır (gerçek figürler üretilir)
  3. Faz9_Word_Olustur.ipynb yeniden çalıştır (figürler Word dosyalarına eklenir)
  4. [PLACEHOLDER] ve [dummy data] bölümlerini gerçek bulgularla doldur
